# 01 — Analyse Exploratoire et Prétraitements Chimiométriques 🌱

**Data Challenge :** Chimiométrie 2027 (SFDS)
**Auteurs :** Abdelouahad ALLA & Marcos FUNEZ FAMOSO (UMons)

## 1. Imports et Configuration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import ChemometricPreprocessor, clean_physical_outliers, extract_modalities

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 2. Chargement des Données et Nettoyage

In [ ]:
df_raw = pd.read_excel('../data/Calibration_set_challenge.xlsx')
print(f"Échantillons : {df_raw.shape[0]} | Colonnes : {df_raw.shape[1]}")

df = clean_physical_outliers(df_raw, is_train=True)
modalities = extract_modalities(df)

y = modalities['y']
X_nir = modalities['X_nir']
X_morpho = modalities['X_morpho']
X_color = modalities['X_color']
wavelengths = modalities['wavelengths']

print(f"Variétés disponibles (250 / classe) : {np.unique(y)}")

## 3. Visualisation des Spectres NIR : Bruts vs Prétraités (SNV + Savitzky-Golay)

In [ ]:
prep = ChemometricPreprocessor(apply_snv=True, sg_window=15, sg_poly=2, sg_deriv=1)
X_nir_prep = prep.fit_transform(X_nir)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Spectres bruts
for i in range(0, len(df), 20):
    axes[0].plot(wavelengths, X_nir[i], alpha=0.3, lw=0.8)
axes[0].set_title("Spectres NIR Bruts (Diffusion et dérives optiques)")
axes[0].set_xlabel("Longueur d'onde (nm)")
axes[0].set_ylabel("Absorbance brute")

# Spectres prétraités
for i in range(0, len(df), 20):
    axes[1].plot(wavelengths, X_nir_prep[i], alpha=0.3, lw=0.8)
axes[1].set_title("Spectres Prétraités (SNV + Savitzky-Golay 1ère dérivée)")
axes[1].set_xlabel("Longueur d'onde (nm)")
axes[1].set_ylabel("Signal corrigé")

plt.tight_layout()
plt.show()

## 4. Distributions Morphologiques et Colorimétriques

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df, x='Variety', y='volume [mm^3]', ax=axes[0], palette='Set2')
axes[0].set_title("Volume par variété de pois (mm³)")
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=df, x='Variety', y='A', ax=axes[1], palette='Pastel1')
axes[1].set_title("Composante couleur A* (axe Vert-Rouge CIELAB)")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Analyse de l'Effet de Lot (Batches)

In [ ]:
ct = pd.crosstab(df['Variety'], df['Batch'])
plt.figure(figsize=(11, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')
plt.title("Répartition des variétés par lot (Batch)")
plt.xlabel("Identifiant du Lot")
plt.ylabel("Variété")
plt.tight_layout()
plt.show()